# N0　MQTT 即時資料示範（第 1 節）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RealKazuya/ai-manufacturing-lab/blob/main/notebooks/N0_mqtt_live.ipynb)

〈AI賦能製造：從工廠數據到AI決策〉課程筆記本。資料全部是教學用虛構（恆準精密工業）。

**用法**：由上往下逐格執行（Shift＋Enter）。要改的參數都在「參數」那一格。

講師在 Colab 執行這本，模擬 VMC-01～03 每 2 秒發布一次主軸負載、溫度、振動與狀態。學員用手機打開 HiveMQ 網頁客戶端，訂閱同一個主題就看得到。

**注意**：這是公開測試伺服器，**任何人都能訂閱**。只放虛構資料。這正是第 1 節 OT 資安要講的事。

In [ ]:
!pip -q install paho-mqtt qrcode[pil]

In [ ]:
# ── 環境設定（每次開啟先執行這一格）──
import sys, os
IN_COLAB = "google.colab" in sys.modules
LOCAL_DATA = os.path.join("..", "data")
DATA = LOCAL_DATA if os.path.isdir(LOCAL_DATA) else "https://raw.githubusercontent.com/RealKazuya/ai-manufacturing-lab/main/data/"
def path(p):
    return os.path.join(DATA, p) if DATA == LOCAL_DATA else DATA + p
print("資料來源：", "本機" if DATA == LOCAL_DATA else "GitHub", "｜Colab：", IN_COLAB)

## 參數

In [ ]:
import random, string
BROKER = "broker.hivemq.com"   # 備用：test.mosquitto.org
PORT = 1883
CODE = "".join(random.choices(string.ascii_lowercase + string.digits, k=6))   # 隨機碼，避免和別人撞主題
TOPIC = "hengzhun-demo/%s/#" % CODE
SECONDS = 900                  # 發布多久（秒）
print("訂閱主題：", TOPIC)

## 開始發布（背景執行，可以繼續跑下面的格）

In [ ]:
import json, time, threading, math
import paho.mqtt.client as mqtt
state = {"alarm": False, "stop": False}
def publisher():
    cli = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    cli.connect(BROKER, PORT, 60); cli.loop_start()
    t0 = time.time()
    while time.time() - t0 < SECONDS and not state["stop"]:
        for m, base in (("VMC-01", 1.1), ("VMC-02", 2.4), ("VMC-03", 1.2)):
            alarm = state["alarm"] and m == "VMC-02"
            msg = {"ts": time.strftime("%H:%M:%S"), "machine": m,
                   "spindle_load_pct": 0.0 if alarm else round(random.uniform(35, 85), 1),
                   "spindle_temp": round(36 + random.gauss(0, 0.3) + (6 if alarm else 0), 1),
                   "vibration_rms": round(base * random.uniform(0.94, 1.06) * (1.9 if alarm else 1), 2),
                   "state": "ALARM" if alarm else "RUN"}
            cli.publish("hengzhun-demo/%s/%s" % (CODE, m), json.dumps(msg, ensure_ascii=False))
        time.sleep(2)
    cli.loop_stop(); cli.disconnect()
threading.Thread(target=publisher, daemon=True).start()
print("發布中。網頁客戶端：https://www.hivemq.com/demos/websocket-client/ ，Host 填", BROKER, "，Port 8884（SSL），訂閱", TOPIC)

In [ ]:
import qrcode
from IPython.display import display
display(qrcode.make("https://www.hivemq.com/demos/websocket-client/"))
print("訂閱主題（請抄這一行）：", TOPIC)

## 把 VMC-02 切成警報（再執行一次就切回來）

In [ ]:
state["alarm"] = not state["alarm"]; print("VMC-02 警報：", state["alarm"])

## 停止發布

In [ ]:
state["stop"] = True; print("已停止")